In [2]:
import time
import pandas as pd
from bs4 import BeautifulSoup, Comment
from seleniumbase import SB

# ──────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────
URL = "https://fbref.com/en/comps/9/stats/Premier-League-Stats"
OUTPUT_CSV = "premier_league_players11.csv"
OUTPUT_EXCEL = "premier_league_players11.xlsx"


print("=" * 60)
print("  FBref Scraper - Premier League Player Stats")
print("  Using SeleniumBase UC Mode (Cloudflare Bypass)")
print("=" * 60)


def scrape_fbref():

    print("\n Launching stealth browser...")
    with SB(uc=True, headless=False) as sb:

        print(f" Navigating to FBref...")
        sb.uc_open_with_reconnect(URL, reconnect_time=5)

        print(" Waiting for Cloudflare challenge...")
        time.sleep(5)

        try:
            sb.uc_gui_click_captcha()
            print(" Cloudflare challenge passed!")
        except Exception:
            print("  No captcha found (may have auto-resolved)")

        print(" Waiting for stats table to load...")
        time.sleep(3)

        page_source = sb.get_page_source()
        print(f" Page size: {len(page_source):,} characters")

        if len(page_source) < 50000:
            print("⚠️  Page may not have loaded fully. Waiting more...")
            time.sleep(10)
            page_source = sb.get_page_source()
            print(f" Page size after wait: {len(page_source):,} characters")


        print("\n� Parsing HTML...")
        soup = BeautifulSoup(page_source, "html.parser")


        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            if "table" in comment:
                comment_soup = BeautifulSoup(comment, "html.parser")
                for table in comment_soup.find_all("table"):
                    comment.replace_with(table)


        soup = BeautifulSoup(str(soup), "html.parser")

        # ── Step 4: Find the standard stats table ──
        # Look for the table with id containing "stats_standard"
        stats_table = soup.find("div", {"id": lambda x: x and "all_stats_standard" in x})

  

        if not stats_table:
            print("❌ Could not find the player stats table!")
            print("   Saving raw HTML for debugging...")
            with open("fbref_debug.html", "w", encoding="utf-8") as f:
                f.write(page_source)
            print("   Saved to fbref_debug.html")
            return None
        

        print(f"✅ Found stats table: id='{stats_table.get('id', 'N/A')}'")

        # ── Step 5: Parse the table into a DataFrame ──
        print("� Converting table to DataFrame...")

        # Use pandas to read the HTML table
        df = pd.read_html(str(stats_table))[0]

        # ── Step 5.1: Manually extract links ──
        print("   Extracting player profile links...")
        
        # FBref often hides tables in comments. Let's find the standard stats table.
        table = soup.find("table", {"id": "stats_standard"})
        
        if not table:
            print(" 🔍 Table not found in main DOM, searching in comments...")
            for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
                if 'id="stats_standard"' in comment:
                    inner_soup = BeautifulSoup(comment, "html.parser")
                    table = inner_soup.find("table", {"id": "stats_standard"})
                    if table:
                        print(" ✅ Found table in comments!")
                        break

        player_links = []
        matches_links = []
        
        if table:
            # Extract player links
            player_cells = table.find_all("td", {"data-stat": "player"})
            for cell in player_cells:
                a_tag = cell.find("a")
                if a_tag and "href" in a_tag.attrs:
                    player_links.append("https://fbref.com" + a_tag["href"])
                else:
                    player_links.append("")
            
            # Extract matches links
            matches_cells = table.find_all("td", {"data-stat": "matches"})
            for cell in matches_cells:
                a_tag = cell.find("a")
                if a_tag and "href" in a_tag.attrs:
                    matches_links.append("https://fbref.com" + a_tag["href"])
                else:
                    matches_links.append("")
        else:
            print(" ❌ Could not find 'stats_standard' table in DOM or comments.")



        # Handle multi-level column headers
        if isinstance(df.columns, pd.MultiIndex):
            # Flatten multi-level columns
            df.columns = [
                f"{col[0]}_{col[1]}" if col[0] != col[1] and col[0] != "" else col[1]
                for col in df.columns
            ]

        # Remove summary/separator rows (rows where Player is NaN or equals the header)
        if "Player" in df.columns:
            df = df[df["Player"].notna()]
            df = df[df["Player"] != "Player"]
        elif any("Player" in str(c) for c in df.columns):
            player_col = [c for c in df.columns if "Player" in str(c)][0]
            df = df[df[player_col].notna()]
            df = df[df[player_col] != "Player"]

        # Assign links
        if len(player_links) == len(df):
            df["Player_URL"] = player_links
            df["Matches_URL"] = matches_links
        else:
            print(f"⚠️ Warning: Link count ({len(player_links)}) does not match record count ({len(df)}).")

        df = df.reset_index(drop=True)

        print(f"\n✅ Scraped {len(df)} player records!")
        print(f"   Columns ({len(df.columns)}):")
        for col in df.columns:
            print(f"     - {col}")

        return df


# ──────────────────────────────────────────────
# Main execution
# ──────────────────────────────────────────────
try:
    df = scrape_fbref()

    if df is not None and len(df) > 0:
        # Show preview
        print("\n" + "=" * 60)
        print("  Preview (first 5 rows):")
        print("=" * 60)
        print(df.head().to_string())

        # Save CSV
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        print(f"\n💾 Saved CSV: {OUTPUT_CSV}")

        # Save Excel
        try:
            df.to_excel(OUTPUT_EXCEL, index=False, sheet_name="Player Stats")
            print(f"📊 Saved Excel: {OUTPUT_EXCEL}")
        except ImportError:
            print("⚠️  openpyxl not installed - skipping Excel")
            print("   pip install openpyxl")

        print(f"\n✅ {len(df)} players × {len(df.columns)} columns")
        print("   Ready for PCA analysis!")
    else:
        print("\n❌ No data was scraped. Check the debug HTML file.")

except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()
    print("\n💡 Troubleshooting:")
    print("   1. Make sure Chrome is installed")
    print("   2. Try running with headless=False to see the browser")
    print("   3. You may need to manually solve the CAPTCHA")

print("\n" + "=" * 60)
print("  Done!")
print("=" * 60)


  FBref Scraper - Premier League Player Stats
  Using SeleniumBase UC Mode (Cloudflare Bypass)



 Launching stealth browser...
 Navigating to FBref...
 Waiting for Cloudflare challenge...
 Cloudflare challenge passed!
 Waiting for stats table to load...
 Page size: 2,494,729 characters

� Parsing HTML...
✅ Found stats table: id='all_stats_standard'
� Converting table to DataFrame...


C:\Users\DZ Laptops\AppData\Local\Temp\ipykernel_26100\2623792440.py:85: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(stats_table))[0]


   Extracting player profile links...

✅ Scraped 535 player records!
   Columns (27):
     - Unnamed: 0_level_0_Rk
     - Unnamed: 1_level_0_Player
     - Unnamed: 2_level_0_Nation
     - Unnamed: 3_level_0_Pos
     - Unnamed: 4_level_0_Squad
     - Unnamed: 5_level_0_Age
     - Unnamed: 6_level_0_Born
     - Playing Time_MP
     - Playing Time_Starts
     - Playing Time_Min
     - Playing Time_90s
     - Performance_Gls
     - Performance_Ast
     - Performance_G+A
     - Performance_G-PK
     - Performance_PK
     - Performance_PKatt
     - Performance_CrdY
     - Performance_CrdR
     - Per 90 Minutes_Gls
     - Per 90 Minutes_Ast
     - Per 90 Minutes_G+A
     - Per 90 Minutes_G-PK
     - Per 90 Minutes_G+A-PK
     - Unnamed: 24_level_0_Matches
     - Player_URL
     - Matches_URL

  Preview (first 5 rows):
  Unnamed: 0_level_0_Rk Unnamed: 1_level_0_Player Unnamed: 2_level_0_Nation Unnamed: 3_level_0_Pos Unnamed: 4_level_0_Squad Unnamed: 5_level_0_Age Unnamed: 6_level_0_Born Playin

In [3]:
import pandas as pd
from bs4 import BeautifulSoup
from seleniumbase import SB
import time
import os

INPUT_CSV = "premier_league_players11.csv"
OUTPUT_CSV = "premier_league_players_with_photos.csv"

def extract_photo_from_html(html):
    """
    Parses the FBref profile HTML to find the headshot image.
    """
    soup = BeautifulSoup(html, 'html.parser')
    meta_div = soup.find('div', id='meta')
    if meta_div:
        img_tag = meta_div.find('img')
        if img_tag and 'src' in img_tag.attrs:
            img_url = img_tag['src']
            if "headshots" in img_url and "silhouette" not in img_url:
                return img_url
    return None

def main():
    print("="*60)
    print("  FBref Player Photo Scraper (SeleniumBase UC Mode)")
    print("="*60)
    
    if not os.path.exists(INPUT_CSV):
        print(f"❌ '{INPUT_CSV}' introuvable.")
        return
        
    df = pd.read_csv(INPUT_CSV)
    
    url_cols = [c for c in df.columns if 'Player_URL' in str(c)]
    if not url_cols:
        print("❌ Colonne 'Player_URL' introuvable.")
        return
    url_col = url_cols[0]
    
    photo_urls = []
    start_index = 0
    if os.path.exists("progress_" + OUTPUT_CSV):
        prog_df = pd.read_csv("progress_" + OUTPUT_CSV)
        if 'Image_URL' in prog_df.columns:
            photo_urls = list(prog_df['Image_URL'].fillna(""))
            start_index = len(photo_urls)
            print(f"🔄 Reprise à partir du joueur {start_index+1}...")
    
    # Use SeleniumBase to bypass Cloudflare
    print("\n🚀 Lancement du navigateur furtif (veuillez ne pas le fermer)...")
    
    with SB(uc=True, headless=False) as sb:
        for i in range(start_index, len(df)):
            row = df.iloc[i]
            url = row[url_col]
            name_cols = [c for c in df.columns if 'Player' in str(c) and 'URL' not in str(c)]
            name = row[name_cols[0]] if name_cols else f"Joueur {i}"
            
            if pd.isna(url) or not str(url).startswith('http') or url == 'Player_URL':
                photo_urls.append("")
                continue
                
            print(f"[{i+1}/{len(df)}] Scraping photo de {name}... ", end="", flush=True)
            
            try:
                # Open URL
                sb.uc_open_with_reconnect(url, reconnect_time=2)
                
                # Check for Cloudflare challenge and wait if necessary
                time.sleep(1.5) # Allow DOM to load
                
                # Get the HTML source and parse it
                page_source = sb.get_page_source()
                photo_url = extract_photo_from_html(page_source)
                
                if photo_url:
                    print("✅ Trouvée")
                    photo_urls.append(photo_url)
                else:
                    # Retry once with a bit more waiting just in case it was a slow load
                    time.sleep(2)
                    page_source = sb.get_page_source()
                    photo_url = extract_photo_from_html(page_source)
                    if photo_url:
                        print("✅ Trouvée (après délai)")
                        photo_urls.append(photo_url)
                    else:
                        print("❌ Aucune")
                        photo_urls.append("")
                        
            except Exception as e:
                print(f"❌ Erreur: {str(e)[:30]}")
                photo_urls.append("")
            
            # Save checkpoint
            if (i + 1) % 10 == 0:
                temp_df = df.iloc[:i+1].copy()
                temp_df['Image_URL'] = photo_urls
                temp_df.to_csv("progress_" + OUTPUT_CSV, index=False)
                
    df['Image_URL'] = photo_urls
    
    if os.path.exists("progress_" + OUTPUT_CSV):
        os.remove("progress_" + OUTPUT_CSV)
        
    have_photos = df[df['Image_URL'].str.len() > 0]
    print(f"\n✅ Terminé ! Photos trouvées pour {len(have_photos)} joueurs sur {len(df)}.")
    
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f"💾 Sauvegardé sous : {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


  FBref Player Photo Scraper (SeleniumBase UC Mode)

🚀 Lancement du navigateur furtif (veuillez ne pas le fermer)...
[1/535] Scraping photo de Brenden Aaronson... ❌ Aucune
[2/535] Scraping photo de Zach Abbott... ✅ Trouvée
[3/535] Scraping photo de Tammy Abraham... ✅ Trouvée
[4/535] Scraping photo de Joshua Acheampong... ✅ Trouvée
[5/535] Scraping photo de Tyler Adams... ✅ Trouvée
[6/535] Scraping photo de Tosin Adarabioyo... ✅ Trouvée
[7/535] Scraping photo de Simon Adingra... ✅ Trouvée
[8/535] Scraping photo de Amine Adli... ✅ Trouvée
[9/535] Scraping photo de Emmanuel Agbadou... ✅ Trouvée
[10/535] Scraping photo de Nayef Aguerd... ✅ Trouvée
[11/535] Scraping photo de Ola Aina... ✅ Trouvée
[12/535] Scraping photo de Rayan Aït-Nouri... ✅ Trouvée
[13/535] Scraping photo de Kristoffer Ajer... ✅ Trouvée
[14/535] Scraping photo de Nathan Aké... ✅ Trouvée
[15/535] Scraping photo de Carlos Alcaraz... ✅ Trouvée
[16/535] Scraping photo de Omar Alderete... ✅ Trouvée
[17/535] Scraping photo de 